<a href="https://colab.research.google.com/github/AkankshaB123/python/blob/main/Price_Simulator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [18]:
import pandas as pd
import statsmodels.api as sm

# -----------------------------
# 1. Load data
# -----------------------------
# Replace with your actual file
# Load the raw data into 'raw_df' to preserve the original DataFrame state
raw_df = pd.read_csv("/content/sample_data/Case_Study_Urgency_Message_Data.xlsx - Aggregated (1).csv", low_memory=False)
# For compatibility, df still points to a copy of raw_df, but subsequent operations should ideally use raw_df directly or its copies
df = raw_df.copy()

In [19]:
df.head(2)

,#,ADR_USD,Unnamed: 2,hotel_id,city_id,City,star_rating,accommadation_type_name,chain_hotel,booking_date,...,Unnamed: 66,Unnamed: 67,Unnamed: 68,Unnamed: 69,Unnamed: 70,Unnamed: 71,Unnamed: 72,Unnamed: 73,Unnamed: 74,Unnamed: 75
0,"17,800",4.26,0.13%,"582,528","9,395",City_A,2.0,Hotel,non-chain,12/2/2016,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,"21,585",4.62,0.15%,"582,528","9,395",City_A,2.0,Hotel,non-chain,12/23/2016,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [20]:
df.columns

Index(['#', 'ADR_USD', 'Unnamed: 2', 'hotel_id', 'city_id', 'City',
       'star_rating', 'accommadation_type_name', 'chain_hotel', 'booking_date',
       'checkin_date', 'checkout_date', 'Stay_duration', 'Week_num',
       'Month_no', 'Date_Diff', 'Date_Range', 'Month_Year', 'Week_name',
       'booking_type', 'Rating_bucket', 'Price bucket', 'Week_day_no',
       'Unnamed: 23', 'Assumption: Every line item is a unique booking',
       'Unnamed: 25', 'Unnamed: 26', 'Unnamed: 27', 'Unnamed: 28',
       'Unnamed: 29', 'Unnamed: 30', 'Unnamed: 31', 'Unnamed: 32',
       'Unnamed: 33', 'Unnamed: 34', 'Unnamed: 35',
       'Core Objective: Should Agoda implement urgency messaging based on how prices behave as check-in approaches? If yes, how and where? --> Yes, but selectively',
       'Unnamed: 37', 'Unnamed: 38', 'Unnamed: 39', 'Unnamed: 40',
       'Unnamed: 41', 'Unnamed: 42', 'Unnamed: 43', 'Unnamed: 44',
       'Unnamed: 45', 'Unnamed: 46', 'Unnamed: 47', 'Unnamed: 48',
       'Unnam

In [51]:
import pandas as pd
import statsmodels.api as sm

# -----------------------------
# 2. Select only needed columns
# -----------------------------
# Create df_for_modeling from raw_df to ensure it always uses the original data
df_for_modeling = raw_df[['ADR_USD', 'star_rating', 'Date_Diff', 'city_id', 'City', 'Price bucket', 'Date_Range', 'checkin_date']].copy()

# Ensure ADR_USD is numeric by removing commas and converting to float
df_for_modeling['ADR_USD'] = df_for_modeling['ADR_USD'].replace({',': ''}, regex=True)
df_for_modeling['ADR_USD'] = pd.to_numeric(df_for_modeling['ADR_USD'], errors='coerce')

# Ensure star_rating and Date_Diff are numeric
df_for_modeling['star_rating'] = pd.to_numeric(df_for_modeling['star_rating'], errors='coerce')
df_for_modeling['Date_Diff'] = pd.to_numeric(df_for_modeling['Date_Diff'], errors='coerce')

# Convert checkin_date to datetime
df_for_modeling['checkin_date'] = pd.to_datetime(df_for_modeling['checkin_date'], errors='coerce')

# Drop missing values if any (including those from ADR_USD and checkin_date conversion)
df_for_modeling = df_for_modeling.dropna()

In [35]:
# Inspect the 'Date_Range' column
print("Unique values in 'Date_Range':\n", df_for_modeling['Date_Range'].unique())
print("\nValue counts for 'Date_Range':\n", df_for_modeling['Date_Range'].value_counts())

Unique values in 'Date_Range':
 ['0-10' '21-30' '31-40' '51-60' '11-20' '41-50' '-1']

Value counts for 'Date_Range':
 Date_Range
0-10     27036
11-20     8189
21-30     5345
31-40     3725
41-50     2725
51-60     2041
-1           3
Name: count, dtype: int64


In [57]:
import numpy as np
import pandas as pd
import statsmodels.api as sm

# Start with a fresh copy of the DataFrame with selected columns
df = df_for_modeling.copy()

# --- DEBUGGING OUTPUT ---
print("--- Debugging Data Loss ---")
print("Original df_for_modeling shape:", df_for_modeling.shape)
print("Shape after initial copy:", df.shape)

# Ordinal mapping for 'Price bucket'
price_bucket_mapping = {'Luxury': 2, 'Budget friendly': 1, 'Low': 0, 'Budget-Friendly': 1}
df['Price_bucket_encoded'] = df['Price bucket'].map(price_bucket_mapping)

unmapped_price_buckets = df[df['Price_bucket_encoded'].isna()]['Price bucket'].unique()
print(f"Unmapped 'Price bucket' values (introducing NaNs): {unmapped_price_buckets}")
print(f"NaNs in 'Price_bucket_encoded' before dropna: {df['Price_bucket_encoded'].isna().sum()}")

# Drop rows where Price_bucket_encoded resulted in NaN due to unmapped categories
df = df.dropna(subset=['Price_bucket_encoded'])
print("Shape after dropping NaNs from encoded columns: (price_bucket)", df.shape)
# --- END DEBUGGING OUTPUT ---

# Perform one-hot encoding on the 'City' column with drop_first=True
df = pd.get_dummies(df, columns=['City'], drop_first=True, dtype=int)

# Perform one-hot encoding on the 'Date_Range' column with drop_first=True
df = pd.get_dummies(df, columns=['Date_Range'], drop_first=True, dtype=int)

# Update X to include the one-hot encoded city and Date_Range columns, and Price_bucket
X = df[['star_rating', 'Date_Diff', 'Price_bucket_encoded'] +
       [col for col in df.columns if 'City_' in col] +
       [col for col in df.columns if 'Date_Range_' in col]]

# Apply logarithmic transformation to ADR_USD and then drop NaNs
y = np.log(pd.to_numeric(df['ADR_USD'], errors='coerce')).dropna()

# After potentially dropping rows in y, ensure X and y still have matching indices and lengths
X = X.loc[y.index]

# Convert any boolean columns in X to integer type for statsmodels compatibility
for col in X.select_dtypes(include='bool').columns:
    X[col] = X[col].astype(int)

print("--- DEBUGGING Y dtypes ---")
print(y.dtypes)
print("--------------------------")

print("--- DEBUGGING X dtypes before add_constant ---")
print(X.dtypes)
print("--------------------------------------------")

# Add a constant (intercept) to the model since we are now using drop_first=True
X = sm.add_constant(X)

print("--- DEBUGGING X dtypes after add_constant ---")
print(X.dtypes)
print("---------------------------------------------")

# Refit the regression model with the updated X and transformed y
model = sm.OLS(y, X).fit()

print('\n=== Model Summary (Log Transformed ADR_USD, drop_first=True) ===')
print(model.summary())

print('\n=== Coefficients (Log Transformed ADR_USD, drop_first=True) ===')
coefficients = model.params
print(coefficients)

--- Debugging Data Loss ---
Original df_for_modeling shape: (49064, 8)
Shape after initial copy: (49064, 8)
Unmapped 'Price bucket' values (introducing NaNs): []
NaNs in 'Price_bucket_encoded' before dropna: 0
Shape after dropping NaNs from encoded columns: (price_bucket) (49064, 9)
--- DEBUGGING Y dtypes ---
float64
--------------------------
--- DEBUGGING X dtypes before add_constant ---
star_rating             float64
Date_Diff                 int64
Price_bucket_encoded      int64
City_City_B               int64
City_City_C               int64
City_City_D               int64
City_City_E               int64
Date_Range_0-10           int64
Date_Range_11-20          int64
Date_Range_21-30          int64
Date_Range_31-40          int64
Date_Range_41-50          int64
Date_Range_51-60          int64
dtype: object
--------------------------------------------
--- DEBUGGING X dtypes after add_constant ---
const                   float64
star_rating             float64
Date_Diff             

In [59]:
import numpy as np
import pandas as pd
import statsmodels.api as sm

def predict_adr_usd(
    star_rating,
    date_diff,
    city,
    price_bucket,
    date_range
):
    # Ensure mapping is available (defined in 9eccec92)
    price_bucket_mapping = {'Luxury': 2, 'Budget friendly': 1, 'Low': 0, 'Budget-Friendly': 1}

    # Initialize a dictionary for the new input
    input_data = {
        'star_rating': [star_rating],
        'Date_Diff': [date_diff],
        'Price_bucket_encoded': [price_bucket_mapping.get(price_bucket, None)]
    }

    # Create a DataFrame for the new input
    input_df = pd.DataFrame(input_data)

    all_feature_names = model.params.index.tolist()

    # Determine reference categories (lexicographically first from unique values for get_dummies with drop_first=True)
    # This assumes consistent sorting with how get_dummies was applied during training.
    # Get unique values from the original df_for_modeling to ensure consistency
    reference_city = sorted(df_for_modeling['City'].unique())[0] # Assuming 'City_A' is dropped
    reference_date_range = sorted(df_for_modeling['Date_Range'].unique())[0] # Assuming '-1' is dropped

    # Handle City one-hot encoding (drop_first=True)
    city_dummies_in_model = [col for col in all_feature_names if col.startswith('City_')]
    for col in city_dummies_in_model:
        input_df[col] = 0
    if city != reference_city:
        city_col_name = f'City_{city}'
        # Add the city column if it exists in the model's features and set to 1
        if city_col_name in all_feature_names:
            input_df[city_col_name] = 1
        else:
            # If the city is not in the model's features, it's treated as the reference category
            pass

    # Handle Date_Range one-hot encoding (drop_first=True)
    date_range_dummies_in_model = [col for col in all_feature_names if col.startswith('Date_Range_')]
    for col in date_range_dummies_in_model:
        input_df[col] = 0
    if date_range != reference_date_range:
        date_range_col_name = f'Date_Range_{date_range}'
        # Add the date_range column if it exists in the model's features and set to 1
        if date_range_col_name in all_feature_names:
            input_df[date_range_col_name] = 1
        else:
            # If the date range is not in the model's features, it's treated as the reference category
            pass

    # Add any missing one-hot encoded columns with value 0 to match the model's expected input
    for col in all_feature_names:
        if col not in input_df.columns and col != 'const':
            input_df[col] = 0

    # Check for Price_bucket_encoded validity
    if input_df['Price_bucket_encoded'].isnull().any():
        print(f"Warning: Price bucket '{price_bucket}' not recognized. Prediction might be inaccurate.")
        return None

    try:
        # Add constant term for prediction if the model was trained with one
        # Drop 'const' from model.params.index if it's there before creating X_predict_data
        model_features_without_const = model.params.index.drop('const', errors='ignore')
        X_predict_data = input_df[model_features_without_const].copy()
        X_predict = sm.add_constant(X_predict_data, has_constant='add')

        # Align the columns of X_predict with the model's expected columns
        X_predict = X_predict[model.params.index]

        # Predict log(ADR_USD)
        predicted_log_adr = model.predict(X_predict)[0]

        # Transform back to original scale (ADR_USD)
        predicted_adr = np.exp(predicted_log_adr)
        return predicted_adr
    except KeyError as e:
        print(f"Error aligning features for prediction: {e}. Check if all required features are present and correctly named.")
        return None
    except Exception as e:
        print(f"An unexpected error occurred during prediction: {e}")
        return None

## Interactive Prediction Input

Use the form below to enter your desired values and get a real-time price prediction.

In [61]:
# @title Enter Values for Prediction
star_rating_input = 5 # @param {type:"number"}
date_diff_input = 10 # @param {type:"integer", title: "X days before checkin"}
city_input = "City_A" # @param ['City_A', 'City_B', 'City_C', 'City_D', 'City_E'] {type:"string"}
price_bucket_input = "Luxury" # @param ['Luxury', 'Budget friendly', 'Low'] {type:"string"}
date_range_input = "0-10" # @param ['0-10', '11-20', '21-30', '31-40', '41-50', '51-60', '-1'] {type:"string"}

print("--- Interactive Prediction ---")
predicted_adr_interactive = predict_adr_usd(
    star_rating=star_rating_input,
    date_diff=date_diff_input,
    city=city_input,
    price_bucket=price_bucket_input,
    date_range=date_range_input
)

if predicted_adr_interactive is not None:
    print(f"Predicted ADR_USD for {city_input}, {star_rating_input} stars, {date_diff_input} days, {price_bucket_input} price bucket, {date_range_input} date range:\n${predicted_adr_interactive:.2f}")
else:
    print("Could not make a prediction with the provided inputs. Please check the city, price bucket, and date range values.")

--- Interactive Prediction ---
Predicted ADR_USD for City_A, 5 stars, 10 days, Luxury price bucket, 0-10 date range:
$346.95
